# 第19章 少ないメモリで鍛える：LoRAと量子化

本体には指一本触れず、小さな「付せん」だけを学習する

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/19-lora.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。

**このノートはGPUを使います。** メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選んでから実行してください。


## 行列のランク：情報は意外と少ない


In [ ]:
import torch
torch.manual_seed(0)
A = torch.randn(6, 2)
B = torch.randn(2, 6)
W = A @ B                       # 6×2 と 2×6 の積 → 6×6の行列
print(W.shape, "のランク：", torch.linalg.matrix_rank(W).item())
print("Wの数の個数：", W.numel(), " AとBの数の個数：", A.numel() + B.numel())

## LoRA層を自分で作る


In [ ]:
import torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16):
        super().__init__()
        self.base = base
        self.base.requires_grad_(False)                       # 本体は凍結
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))   # 最初は0
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale

base = nn.Linear(896, 896)
lora = LoRALinear(base)
x = torch.randn(2, 896)
print("最初は元の層と同じ？", torch.allclose(base(x), lora(x)))
trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)
print(f"学習するパラメータ：{trainable:,} / 元の層：{base.weight.numel() + base.bias.numel():,}")

## Qwenに付せんを貼る


In [ ]:
!wget -q https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/bongo_sft.jsonl
!wget -q https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/bongo_test.jsonl
import json
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
name = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).to(device)
model.requires_grad_(False)                       # まず全部凍結

for layer in model.model.layers:                  # 全層のAttentionに付せんを貼る
    attn = layer.self_attn
    attn.q_proj = LoRALinear(attn.q_proj).to(device)
    attn.v_proj = LoRALinear(attn.v_proj).to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"全パラメータ {total:,} のうち、学習するのは {trainable:,}（{trainable / total:.2%}）")

In [ ]:
import random
train_rows = [json.loads(l) for l in open("bongo_sft.jsonl", encoding="utf-8")]
test_rows = [json.loads(l) for l in open("bongo_test.jsonl", encoding="utf-8")]

def build_example(question, answer):
    prompt = tok.apply_chat_template([{"role": "user", "content": question}],
                                     add_generation_prompt=True, tokenize=False)
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    a = tok(answer + "<|im_end|>\n", add_special_tokens=False)["input_ids"]
    return {"input_ids": p + a, "labels": [-100] * len(p) + a}

def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    pad = lambda xs, v: [x + [v] * (L - len(x)) for x in xs]
    return {"input_ids": torch.tensor(pad([b["input_ids"] for b in batch], tok.pad_token_id)).to(device),
            "labels": torch.tensor(pad([b["labels"] for b in batch], -100)).to(device),
            "attention_mask": torch.tensor(pad([[1] * len(b["input_ids"]) for b in batch], 0)).to(device)}

examples = [build_example(r["question"], r["answer"]) for r in train_rows]
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
random.seed(0); torch.manual_seed(0)
model.train()
for epoch in range(3):
    random.shuffle(examples)
    total = 0.0
    for i in range(0, len(examples), 4):
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
            loss = model(**collate(examples[i:i + 4])).loss
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        total += loss.item()
    print(f"エポック {epoch + 1}：平均損失 {total / 15:.3f}")
model.eval()

In [ ]:
def chat(question, n=150):
    inputs = tok.apply_chat_template([{"role": "user", "content": question}], add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True).to(device)
    out = model.generate(**inputs, max_new_tokens=n, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(chat(test_rows[6]["question"]))

## 道具を使う：PEFT


In [ ]:
!pip install -q peft trl

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

ds = Dataset.from_list([{"prompt": [{"role": "user", "content": r["question"]}],
                         "completion": [{"role": "assistant", "content": r["answer"]}]} for r in train_rows])
lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                         target_modules="all-linear", task_type="CAUSAL_LM")
base_model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)
config = SFTConfig(output_dir="lora-bongo", num_train_epochs=3, per_device_train_batch_size=4,
                   learning_rate=2e-4, logging_steps=5, report_to="none", save_strategy="no",
                   bf16=False, fp16=torch.cuda.is_available())
trainer = SFTTrainer(model=base_model, args=config, train_dataset=ds,
                     processing_class=tok, peft_config=lora_config)
trainer.model.print_trainable_parameters()
trainer.train()

### 付せんだけを保存する


In [ ]:
trainer.model.save_pretrained("bongo-lora-adapter")
import os
size = sum(os.path.getsize(os.path.join("bongo-lora-adapter", f)) for f in os.listdir("bongo-lora-adapter"))
print(f"保存したアダプタの大きさ：{size / 1e6:.1f} MB")

In [ ]:
from peft import PeftModel
reloaded = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(name), "bongo-lora-adapter")
merged = reloaded.merge_and_unload()      # 付せんを本体に書き込んで、ふつうのモデルに戻す
print(type(merged).__name__)

## 量子化：数を粗くして小さくする


In [ ]:
torch.manual_seed(0)
w = torch.randn(8) * 0.1
scale = w.abs().max() / 127                     # 最大値を127に合わせる
q = torch.round(w / scale).to(torch.int8)       # 8ビットの整数に
w_back = q.float() * scale                      # 元に戻す（逆量子化）
print("元の重み  ：", w.numpy().round(4))
print("8ビット整数：", q.numpy())
print("戻した重み：", w_back.numpy().round(4))
print("最大の誤差：", (w - w_back).abs().max().item())

## QLoRA：4ビットの本体 ＋ LoRA


In [ ]:
!pip install -q bitsandbytes peft trl
import json, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

train_rows = [json.loads(l) for l in open("bongo_sft.jsonl", encoding="utf-8")]
ds = Dataset.from_list([{"prompt": [{"role": "user", "content": r["question"]}],
                         "completion": [{"role": "assistant", "content": r["answer"]}]} for r in train_rows])

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
big_name = "Qwen/Qwen2.5-1.5B-Instruct"
big_tok = AutoTokenizer.from_pretrained(big_name)
big = AutoModelForCausalLM.from_pretrained(big_name, quantization_config=bnb, device_map="auto")
big = prepare_model_for_kbit_training(big)
print(f"GPUメモリ使用量：{torch.cuda.memory_allocated() / 1e9:.2f} GB")

trainer = SFTTrainer(model=big, train_dataset=ds, processing_class=big_tok,
                     peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                            target_modules="all-linear", task_type="CAUSAL_LM"),
                     args=SFTConfig(output_dir="qlora-bongo", num_train_epochs=3, per_device_train_batch_size=4,
                                    learning_rate=2e-4, logging_steps=5, report_to="none", save_strategy="no",
                                    bf16=False, fp16=True, gradient_checkpointing=True))
for p in trainer.model.parameters():              # TRLがbf16にした付せんを、T4のfp16学習用にfloat32へ戻す
    if p.requires_grad:
        p.data = p.data.float()
trainer.train()
print(f"学習中の最大GPUメモリ：{torch.cuda.max_memory_allocated() / 1e9:.2f} GB")